# 01 Smoke Test: Synthetic Segmentation Pipeline

This notebook verifies the refactored modular pipeline () with a fast 1-epoch smoke test.

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.vision.segmentation.models import UNetSurrogate
from src.vision.segmentation.data import build_dataloaders
from src.utils.seed import set_seed
from src.utils.visualize import visualize_predictions

set_seed(42)
print("Environment initialized successfully.")

## 1. Load Synthetic Dataloaders

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_loader, val_loader = build_dataloaders(
    batch_size=16,
    image_size=(128, 128),
    num_classes=8,
    train_samples=64,
    val_samples=32,
    seed=42,
)

images, masks = next(iter(train_loader))
print(f"Sample batch shapes: images={images.shape}, masks={masks.shape}")

## 2. Initialize Model & Optimizer

In [ ]:
model = UNetSurrogate(in_channels=3, num_classes=8, up_channels=16).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
print(f"Initialized UNetSurrogate with {sum(p.numel() for p in model.parameters()):,} parameters.")

## 3. Run 1-Epoch Smoke Test

In [ ]:
model.train()
train_loss = 0.0
for imgs, msks in train_loader:
    imgs, msks = imgs.to(device), msks.to(device)
    optimizer.zero_grad()
    preds = model(imgs)
    loss = criterion(preds, msks)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()

model.eval()
val_loss = 0.0
with torch.no_grad():
    for imgs, msks in val_loader:
        imgs, msks = imgs.to(device), msks.to(device)
        preds = model(imgs)
        val_loss += criterion(preds, msks).item()

print(f"Smoke test completed: Train Loss={train_loss/len(train_loader):.4f} | Val Loss={val_loss/len(val_loader):.4f}")

## 4. Visualize Predictions

In [ ]:
fig = visualize_predictions(model, val_loader, num_images=2, device=device, show=False)
print("Visualization generated successfully.")